In [1]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "SparkSQL Example 1"
su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 00:37:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=spark://spark-master:7077 appName=SparkSQL Example 1>

In [2]:
from pyspark.sql.types import StructField, StringType, StructType, IntegerType

data = [
(1, "Alice", 29),
(2, "Bob", 35),
(3, "Charlie", 41)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df = su._spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



In [3]:
from pyspark.sql.types import StructField, StringType, StructType, TimestampType, FloatType
from datetime import datetime

factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("temperature", FloatType(), True),
])

factory_schema

StructType([StructField('id', StringType(), True), StructField('timestamp', TimestampType(), True), StructField('temperature', FloatType(), True)])

In [4]:
df_factory = su._spark.createDataFrame(factory_data, factory_schema)
df_factory.printSchema()

root
 |-- id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: float (nullable = true)



In [5]:
print(f"Factory Data: {df_factory.count()}")

[Stage 0:=============================>                             (1 + 1) / 2]

Factory Data: 10


In [6]:
from pyspark.sql.functions import avg, min, max
agg_df = df_factory.groupBy("id").agg(
    avg("temperature").alias("avg_temperature"),
    min("temperature").alias("min_temperature"),
    max("temperature").alias("max_temperature")
)
agg_df.show()

[Stage 3:=============================>                             (1 + 1) / 2]

+----+-----------------+---------------+---------------+
|  id|  avg_temperature|min_temperature|max_temperature|
+----+-----------------+---------------+---------------+
|M002|69.53333282470703|           68.7|           70.1|
|M003|73.39999898274739|           72.4|           74.6|
|M001|76.72500038146973|           75.3|           78.0|
+----+-----------------+---------------+---------------+



In [7]:
from pyspark.sql.functions import col
df_filtered = df_factory.filter(col("temperature") > 75)
df_filtered.show(truncate=False)

+----+-------------------+-----------+
|id  |timestamp          |temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:00:00|75.3       |
|M001|2026-01-26 08:10:00|76.1       |
|M001|2026-01-26 08:25:00|77.5       |
|M001|2026-01-26 08:40:00|78.0       |
+----+-------------------+-----------+



In [8]:
factory_groupped = df_factory.groupBy(col("id")).count()
factory_groupped.show()

[Stage 8:=============================>                             (1 + 1) / 2]

+----+-----+
|  id|count|
+----+-----+
|M002|    3|
|M003|    3|
|M001|    4|
+----+-----+



In [18]:
# Find the machine with the highest  temperature
df_factory = su._spark.createDataFrame(factory_data, factory_schema)
df_factory = df_factory.orderBy(col("temperature"), ascending=False).limit(1)
df_factory.show()

+----+-------------------+-----------+
|  id|          timestamp|temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:40:00|       78.0|
+----+-------------------+-----------+



In [19]:
su._spark.stop()